# Phase 1.2: Realistic Channel Modeling Demonstration

This notebook demonstrates production-grade channel models per 3GPP specifications:
- 3GPP TR 38.901 TDL channel models (TDL-A/B/C/D/E)
- Jakes' sum-of-sinusoids time-varying fading with Doppler
- Hardware impairments (CFO, SFO, I/Q imbalance, DC offset, phase noise, PA nonlinearity)
- MIMO with spatial correlation

All implementations based on 3GPP specifications with proper citations.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from utils_channel import (
    add_awgn,
    generate_tdl_channel,
    apply_tdl_channel,
    generate_rayleigh_fading_jakes,
    generate_rician_fading_jakes,
    apply_cfo,
    apply_sfo,
    apply_iq_imbalance,
    apply_dc_offset,
    apply_phase_noise,
    apply_pa_nonlinearity,
    generate_mimo_channel,
    apply_mimo_channel,
    TDL_MODELS
)

from run_lte import generate_lte_signal

print('Channel modeling imports successful')

## 1. 3GPP TDL Channel Models

3GPP TR 38.901 TDL (Tapped Delay Line) models replace obsolete ITU profiles.

In [ ]:
# Show all TDL models
print('3GPP TR 38.901 TDL Models:')
print('=' * 80)
for model_name, model_info in TDL_MODELS.items():
    print(f"\n{model_name}: {model_info['description']}")
    print(f"  Number of taps: {len(model_info['delays_normalized'])}")
    print(f"  Typical delay spread: {model_info['typical_ds_ns']} ns")
    if model_name in ['TDL-D', 'TDL-E']:
        k_factor = model_info['k_factors_db'][0]
        print(f"  K-factor (LOS): {k_factor} dB")

In [ ]:
# Generate and visualize TDL-A channel
num_samples = 5000
sample_rate = 15.36e6

channel_coeffs, delays_ns, powers = generate_tdl_channel(
    num_samples=num_samples,
    tdl_model='TDL-A',
    sample_rate=sample_rate,
    doppler_hz=100,
    device='cpu'
)

print(f"Channel coefficients shape: {channel_coeffs.shape}")
print(f"Number of taps: {len(delays_ns)}")
print(f"Total channel power: {sum(powers):.6f}")

# Plot power delay profile
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Power delay profile
axes[0].stem(delays_ns, 10*np.log10(powers), basefmt=' ')
axes[0].set_xlabel('Delay (ns)')
axes[0].set_ylabel('Power (dB)')
axes[0].set_title('TDL-A Power Delay Profile')
axes[0].grid(True)

# Time-varying fading for first tap
t = np.arange(1000) / sample_rate * 1000
h0 = channel_coeffs[0, :1000].cpu().numpy()
axes[1].plot(t, np.abs(h0))
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('|h(t)|')
axes[1].set_title('TDL-A First Tap Time Variation (100 Hz Doppler)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Compare NLOS (TDL-A) vs LOS (TDL-D)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, model_name in enumerate(['TDL-A', 'TDL-D']):
    coeffs, delays, powers = generate_tdl_channel(
        num_samples=1000,
        tdl_model=model_name,
        sample_rate=sample_rate,
        device='cpu'
    )
    
    axes[idx].stem(delays, 10*np.log10(powers), basefmt=' ')
    axes[idx].set_xlabel('Delay (ns)')
    axes[idx].set_ylabel('Power (dB)')
    axes[idx].set_title(f'{model_name}: {TDL_MODELS[model_name]["description"]}')
    axes[idx].grid(True)

plt.tight_layout()
plt.show()

## 2. Jakes' Time-Varying Fading

Jakes' sum-of-sinusoids model for Rayleigh/Rician fading with Doppler effects.

In [ ]:
# Compare static vs time-varying Rayleigh fading
num_samples = 10000

h_static = generate_rayleigh_fading_jakes(num_samples, doppler_freq=0)
h_doppler = generate_rayleigh_fading_jakes(num_samples, doppler_freq=100, sample_rate=15.36e6)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Static fading
t = np.arange(1000) / 15.36e6 * 1000
h_static_np = h_static[:1000].cpu().numpy()
axes[0, 0].plot(t, np.abs(h_static_np))
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('|h(t)|')
axes[0, 0].set_title('Static Rayleigh Fading (no Doppler)')
axes[0, 0].grid(True)

# Time-varying fading
h_doppler_np = h_doppler[:1000].cpu().numpy()
axes[0, 1].plot(t, np.abs(h_doppler_np))
axes[0, 1].set_xlabel('Time (ms)')
axes[0, 1].set_ylabel('|h(t)|')
axes[0, 1].set_title('Time-Varying Rayleigh Fading (100 Hz Doppler)')
axes[0, 1].grid(True)

# Amplitude distributions
axes[1, 0].hist(np.abs(h_static.cpu().numpy()), bins=50, density=True, alpha=0.7)
axes[1, 0].set_xlabel('Amplitude')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Static: Amplitude Distribution')
axes[1, 0].grid(True)

axes[1, 1].hist(np.abs(h_doppler.cpu().numpy()), bins=50, density=True, alpha=0.7)
axes[1, 1].set_xlabel('Amplitude')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Doppler: Amplitude Distribution')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print(f"Static: Mean power = {torch.mean(torch.abs(h_static)**2).item():.6f}")
print(f"Doppler: Mean power = {torch.mean(torch.abs(h_doppler)**2).item():.6f}")

In [ ]:
# Rician fading with different K-factors
k_factors = [0, 3, 6, 10]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, k_db in enumerate(k_factors):
    row, col = idx // 2, idx % 2
    h = generate_rician_fading_jakes(10000, k_factor_db=k_db)
    h_np = h.cpu().numpy()
    
    axes[row, col].hist(np.abs(h_np), bins=50, density=True, alpha=0.7)
    axes[row, col].set_xlabel('Amplitude')
    axes[row, col].set_ylabel('Density')
    axes[row, col].set_title(f'Rician K={k_db} dB (mean={np.mean(np.abs(h_np)):.3f})')
    axes[row, col].grid(True)

plt.tight_layout()
plt.show()

## 3. Hardware Impairments

Realistic RF hardware imperfections per 3GPP specifications.

In [ ]:
# Generate clean LTE signal for testing
lte_result = generate_lte_signal(
    duration_ms=1.0,
    bandwidth_mhz=10.0,
    modulation_scheme='16QAM',
    power_dbm=0.0,
    device='cpu'
)

clean_signal = lte_result['signal']
lte_sample_rate = lte_result['metadata']['sample_rate']

print(f"LTE signal: {len(clean_signal)} samples at {lte_sample_rate/1e6:.2f} MHz")
print(f"Signal power: {torch.mean(torch.abs(clean_signal)**2).item():.6f}")

In [ ]:
# CFO effect on constellation
cfo_values = [0, 1000, 5000, 10000]  # Hz
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, cfo_hz in enumerate(cfo_values):
    row, col = idx // 2, idx % 2
    
    signal_with_cfo = apply_cfo(clean_signal, cfo_hz, lte_sample_rate)
    sig_np = signal_with_cfo.cpu().numpy()[::10]
    
    axes[row, col].scatter(sig_np.real, sig_np.imag, alpha=0.3, s=1)
    axes[row, col].set_xlabel('In-phase')
    axes[row, col].set_ylabel('Quadrature')
    axes[row, col].set_title(f'CFO = {cfo_hz} Hz')
    axes[row, col].grid(True)
    axes[row, col].axis('equal')
    axes[row, col].set_xlim(-2, 2)
    axes[row, col].set_ylim(-2, 2)

plt.tight_layout()
plt.show()

In [ ]:
# I/Q imbalance effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clean signal
clean_np = clean_signal.cpu().numpy()[::10]
axes[0].scatter(clean_np.real, clean_np.imag, alpha=0.3, s=1)
axes[0].set_xlabel('In-phase')
axes[0].set_ylabel('Quadrature')
axes[0].set_title('Clean Signal')
axes[0].grid(True)
axes[0].axis('equal')

# With I/Q imbalance
iq_imbalanced = apply_iq_imbalance(clean_signal, amplitude_imb_db=2.0, phase_imb_deg=10.0)
iq_np = iq_imbalanced.cpu().numpy()[::10]
axes[1].scatter(iq_np.real, iq_np.imag, alpha=0.3, s=1)
axes[1].set_xlabel('In-phase')
axes[1].set_ylabel('Quadrature')
axes[1].set_title('With I/Q Imbalance (2 dB, 10 deg)')
axes[1].grid(True)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# PA nonlinearity (clipping)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amplitude distribution before PA
clean_amp = np.abs(clean_signal.cpu().numpy())
axes[0].hist(clean_amp, bins=50, alpha=0.7, label='Clean')

# After PA nonlinearity
amplified = apply_pa_nonlinearity(clean_signal, input_backoff_db=3.0)
amp_np = np.abs(amplified.cpu().numpy())
axes[0].hist(amp_np, bins=50, alpha=0.7, label='After PA')
axes[0].set_xlabel('Amplitude')
axes[0].set_ylabel('Count')
axes[0].set_title('Amplitude Distribution (PA Clipping Effect)')
axes[0].legend()
axes[0].grid(True)

# AM/AM characteristic
axes[1].scatter(clean_amp[::100], amp_np[::100], alpha=0.3, s=1)
axes[1].plot([0, clean_amp.max()], [0, clean_amp.max()], 'r--', label='Linear')
axes[1].set_xlabel('Input Amplitude')
axes[1].set_ylabel('Output Amplitude')
axes[1].set_title('AM/AM Characteristic (Rapp Model)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 4. MIMO Channels

MIMO channel generation with time-varying fading and spatial correlation.

In [ ]:
# Generate 4x4 MIMO channel
num_tx, num_rx = 4, 4
H = generate_mimo_channel(
    num_tx=num_tx,
    num_rx=num_rx,
    num_samples=5000,
    doppler_hz=100,
    sample_rate=15.36e6
)

print(f"MIMO channel shape: {H.shape}")

# Visualize channel matrix
fig, axes = plt.subplots(num_rx, num_tx, figsize=(16, 12))
fig.suptitle(f'{num_tx}x{num_rx} MIMO Channel Matrix (|H|)', fontsize=16)

for rx_idx in range(num_rx):
    for tx_idx in range(num_tx):
        h_element = H[rx_idx, tx_idx, :1000].cpu().numpy()
        t = np.arange(len(h_element)) / 15.36e6 * 1000
        
        axes[rx_idx, tx_idx].plot(t, np.abs(h_element))
        axes[rx_idx, tx_idx].set_title(f'H[{rx_idx},{tx_idx}]', fontsize=10)
        axes[rx_idx, tx_idx].grid(True, alpha=0.3)
        
        if rx_idx == num_rx - 1:
            axes[rx_idx, tx_idx].set_xlabel('Time (ms)', fontsize=8)
        if tx_idx == 0:
            axes[rx_idx, tx_idx].set_ylabel('|h(t)|', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Realistic Combined Scenario

Complete channel with TDL + CFO + I/Q imbalance + AWGN applied to LTE signal.

In [ ]:
# Apply realistic channel cascade
signal = clean_signal.clone()

print(f"Original signal power: {torch.mean(torch.abs(signal)**2).item():.6f}")

# Step 1: TDL-A multipath fading
signal = apply_tdl_channel(signal, tdl_model='TDL-A', doppler_hz=100, sample_rate=lte_sample_rate)
print(f"After TDL-A:          {torch.mean(torch.abs(signal)**2).item():.6f}")

# Step 2: CFO (5 ppm at 3.5 GHz = 17.5 kHz)
signal = apply_cfo(signal, cfo_hz=17500, sample_rate=lte_sample_rate)
print(f"After CFO:            {torch.mean(torch.abs(signal)**2).item():.6f}")

# Step 3: I/Q imbalance
signal = apply_iq_imbalance(signal, amplitude_imb_db=1.0, phase_imb_deg=5.0)
print(f"After I/Q imbalance:  {torch.mean(torch.abs(signal)**2).item():.6f}")

# Step 4: AWGN (20 dB SNR)
signal = add_awgn(signal, snr_db=20)
print(f"After AWGN:           {torch.mean(torch.abs(signal)**2).item():.6f}")

# Compare constellations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clean_np = clean_signal.cpu().numpy()[::10]
axes[0].scatter(clean_np.real, clean_np.imag, alpha=0.3, s=1)
axes[0].set_xlabel('In-phase')
axes[0].set_ylabel('Quadrature')
axes[0].set_title('Clean LTE Signal')
axes[0].grid(True)
axes[0].axis('equal')

sig_np = signal.cpu().numpy()[::10]
axes[1].scatter(sig_np.real, sig_np.imag, alpha=0.3, s=1)
axes[1].set_xlabel('In-phase')
axes[1].set_ylabel('Quadrature')
axes[1].set_title('After Realistic Channel (TDL + CFO + I/Q + AWGN)')
axes[1].grid(True)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

## Summary

Phase 1.2 demonstrates production-grade channel modeling:

1. 3GPP TR 38.901 TDL models (TDL-A/B/C/D/E) - replaces obsolete ITU profiles
2. Jakes' sum-of-sinusoids model - time-varying fading with Doppler
3. Hardware impairments - CFO, SFO, I/Q imbalance, DC offset, phase noise, PA nonlinearity
4. MIMO channels - time-varying with spatial correlation
5. Realistic cascaded scenarios - multiple effects combined

All implementations based on 3GPP specifications with proper citations in paper/amendment.md.